# Linear Regression

- by using Onehotencoding, StringIndexer, StandardScaler
- using pipeline


In [14]:
import os
import numpy as np
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, StringIndexer, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
spark = SparkSession.builder.appName('HousingModelDocker').getOrCreate()

In [3]:
RECORD_TO_SHOW=2

## Step 1: Create Dataframe

In [4]:
data = spark.read.csv('work/housing_prices.csv', header=True, inferSchema=True)
data.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



In [5]:
data.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

##### Handling NULL values

In [6]:
(
    data
        .select([
            F.count(
                F.when(
                    F.col(c).isNull(), 1
                )
            ).alias(c) for c in data.columns]
        ).show()
)

data = data.dropna(subset='total_bedrooms')

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|        0|       0|                 0|          0|           207|         0|         0|            0|                 0|              0|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+



## Step 2: Create pipeline

In [21]:
training_data, testing_data = data.randomSplit([0.8,0.2], seed=42)
print(f'Training data count: {training_data.count()}')
print(f'Testing data count: {testing_data.count()}')

indexer = StringIndexer(inputCol='ocean_proximity', outputCol='ocean_proximity_index')
encoder = OneHotEncoder(inputCol='ocean_proximity_index', outputCol='ocean_proximity_vec', dropLast=False)


feature_columns = [
    'housing_median_age',
    'total_rooms',
    'total_bedrooms',
    'population',
    'households',
    'median_income',
    'ocean_proximity_vec'
]

assembler = VectorAssembler(inputCols=feature_columns, outputCol='unscaled_features')
scaler = StandardScaler(inputCol='unscaled_features',outputCol='features', withMean=True, withStd=True)
lr = LinearRegression(featuresCol='features', labelCol='median_house_value', regParam=0.001)
pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, lr])
pipeline_model = pipeline.fit(training_data)

Training data count: 16395
Testing data count: 4038


## Step 4: Prediction

In [22]:
prediction_data = pipeline_model.transform(testing_data)

## Evaluation

In [23]:
evaluation = RegressionEvaluator(labelCol='median_house_value',predictionCol='prediction', metricName='mae')
mae = evaluation.evaluate(prediction_data)
print(f'Mean Absolute Error: {mae}')

Mean Absolute Error: 50597.33640580943
